# 🤖 Pepê AI — Fine-tuning com LoRA

Fine-tuning do **Llama 3 8B** com os dados do Pepê usando **QLoRA** via Unsloth.

**Requisitos:** Runtime → Alterar tipo de ambiente de execução → **GPU T4**

| Etapa | Descrição |
|-------|-----------|
| 1 | Instalar dependências |
| 2 | Carregar dataset do GitHub |
| 3 | Configurar modelo + LoRA |
| 4 | Treinar |
| 5 | Exportar GGUF |
| 6 | Testar modelo |


## ⚙️ Etapa 1 — Instalar dependências

In [ ]:
# Verifica GPU
!nvidia-smi

# Instala Unsloth (otimizado para T4)
!pip install unsloth==2024.11.4 -q
!pip install datasets transformers accelerate bitsandbytes trl -q
!pip install -q huggingface_hub

print('✅ Dependências instaladas')

## 📦 Etapa 2 — Carregar dataset do GitHub

In [ ]:
import json
import urllib.request
from datasets import Dataset

# URLs do dataset no GitHub
BASE_URL = 'https://raw.githubusercontent.com/Joao-Matheus-Amorim/pepe-ai/main/training/datasets/'

def load_jsonl_from_url(url: str) -> list[dict]:
    with urllib.request.urlopen(url) as r:
        return [json.loads(line) for line in r.read().decode().splitlines() if line.strip()]

train_data = load_jsonl_from_url(BASE_URL + 'pepe_train.jsonl')
val_data   = load_jsonl_from_url(BASE_URL + 'pepe_val.jsonl')

print(f'✅ Treino:    {len(train_data)} exemplos')
print(f'✅ Validação: {len(val_data)} exemplos')

# Mostra exemplo
print('\n--- Exemplo de treino ---')
for msg in train_data[0]['messages']:
    role = msg['role'].upper()
    content = msg.get('content') or '[tool_call]'
    print(f'[{role}] {content[:120]}')

## 🧠 Etapa 3 — Carregar modelo Llama 3 8B + QLoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
DTYPE = None          # auto-detect (float16 na T4)
LOAD_IN_4BIT = True   # QLoRA — reduz VRAM ~4x

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name   = 'unsloth/Meta-Llama-3-8B-Instruct',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = LOAD_IN_4BIT,
)

print('✅ Modelo carregado')
print(f'   Parâmetros: {model.num_parameters()/1e6:.0f}M')

In [ ]:
# Configuração LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,     # rank — equilíbrio qualidade/velocidade
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha     = 32,     # alpha = 2x rank
    lora_dropout   = 0.05,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',  # economiza VRAM
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA configurado')
print(f'   Parâmetros treináveis: {trainable/1e6:.2f}M ({100*trainable/total:.2f}%)')

## 📝 Etapa 4 — Preparar dataset e treinar

In [ ]:
from unsloth.chat_templates import get_chat_template

# Aplica template de chat do Llama 3
tokenizer = get_chat_template(tokenizer, chat_template='llama-3')

def format_messages(examples):
    texts = []
    for msgs in examples['messages']:
        # Filtra mensagens de tool (não suportadas nativamente no template)
        clean = [m for m in msgs if m.get('role') in ('system', 'user', 'assistant')
                 and m.get('content') is not None]
        text = tokenizer.apply_chat_template(
            clean,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return {'text': texts}

# Converte para HuggingFace Dataset
train_hf = Dataset.from_list(train_data).map(format_messages, batched=True)
val_hf   = Dataset.from_list(val_data).map(format_messages, batched=True)

print(f'✅ Dataset formatado')
print(f'   Exemplo:\n{train_hf[0]["text"][:300]}...')

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model          = model,
    tokenizer      = tokenizer,
    train_dataset  = train_hf,
    eval_dataset   = val_hf,
    dataset_text_field = 'text',
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing        = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,    # batch efetivo = 8
        num_train_epochs            = 3,
        warmup_steps                = 10,
        learning_rate               = 2e-4,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 10,
        evaluation_strategy         = 'steps',
        eval_steps                  = 50,
        save_strategy               = 'steps',
        save_steps                  = 100,
        output_dir                  = '/content/pepe-ft-checkpoints',
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        lr_scheduler_type           = 'cosine',
        seed                        = 42,
        report_to                   = 'none',
    ),
)

print('🚀 Iniciando treinamento...')
trainer_stats = trainer.train()
print(f'\n✅ Treinamento concluído!')
print(f'   Tempo: {trainer_stats.metrics["train_runtime"]:.0f}s')
print(f'   Loss final: {trainer_stats.metrics["train_loss"]:.4f}')

## 💾 Etapa 5 — Exportar modelo GGUF para Ollama

In [ ]:
# Salva adaptadores LoRA
model.save_pretrained('/content/pepe-ft-lora')
tokenizer.save_pretrained('/content/pepe-ft-lora')
print('✅ Adaptadores LoRA salvos em /content/pepe-ft-lora')

# Exporta GGUF Q4_K_M (melhor equilíbrio qualidade/tamanho para Ollama)
print('\n🔄 Exportando GGUF Q4_K_M...')
model.save_pretrained_gguf(
    '/content/pepe-ft-gguf',
    tokenizer,
    quantization_method = 'q4_k_m'
)
print('✅ GGUF exportado em /content/pepe-ft-gguf')

# Lista arquivos gerados
import os
for f in os.listdir('/content/pepe-ft-gguf'):
    size = os.path.getsize(f'/content/pepe-ft-gguf/{f}') / 1e9
    print(f'   {f}: {size:.2f} GB')

In [ ]:
# Faz download do GGUF para o seu PC
from google.colab import files
import os

gguf_dir = '/content/pepe-ft-gguf'
for fname in os.listdir(gguf_dir):
    if fname.endswith('.gguf'):
        print(f'📥 Baixando {fname}...')
        files.download(f'{gguf_dir}/{fname}')
        print('✅ Download iniciado')

## 🧪 Etapa 6 — Testar o modelo fine-tunado

In [ ]:
from unsloth import FastLanguageModel

# Ativa modo inferência
FastLanguageModel.for_inference(model)

SYSTEM_PEPE = (
    'Você é Pepê, um agente de IA pessoal inteligente, direto e eficiente. '
    'Você tem acesso a ferramentas de busca web, clima, visão de tela, execução de comandos, '
    'leitura de arquivos e memória persistente. '
    'Responda sempre em português do Brasil, de forma clara e concisa.'
)

def testar(pergunta: str) -> str:
    msgs = [
        {'role': 'system',    'content': SYSTEM_PEPE},
        {'role': 'user',      'content': pergunta},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    outputs = model.generate(
        input_ids  = inputs,
        max_new_tokens = 256,
        temperature    = 0.7,
        do_sample      = True,
    )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Pega só a resposta do assistant
    return decoded.split('assistant')[-1].strip()

# Testes de personalidade e identidade
perguntas = [
    'Quem é você?',
    'Quem criou você?',
    'Qual a stack do projeto pepe-ai?',
    'O que você consegue fazer?',
    'Qual o próximo passo do projeto?',
]

for p in perguntas:
    print(f'\n[USER] {p}')
    print(f'[PEPÊ] {testar(p)}')

## 🚀 Etapa 7 — Integrar com Ollama (rodar no seu PC)

Após o download do `.gguf`, execute no seu PC:

```bash
# 1. Crie o Modelfile
cat > Modelfile << 'EOF'
FROM ./pepe-ft-unsloth.Q4_K_M.gguf

SYSTEM "Você é Pepê, um agente de IA pessoal inteligente, direto e eficiente. Você tem acesso a ferramentas de busca web, clima, visão de tela, execução de comandos, leitura de arquivos e memória persistente. Responda sempre em português do Brasil, de forma clara e concisa."

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
EOF

# 2. Cria o modelo no Ollama
ollama create pepe-ft -f Modelfile

# 3. Testa
ollama run pepe-ft "Quem é você?"

# 4. Ativa no pepe-ai (edite o .env)
# PEPE_MODEL_PROVIDER=ollama
# PEPE_MODEL_NAME=pepe-ft
```
